# LangGraph Core Concepts
> A step-by-step walkthrough of all six core concept areas using **GPT-4o Mini** and **Google Colab Secrets**.

| Section | Topic |
|---|---|
| 1 | 🔧 Setup & Secrets |
| 2 | 🗃️ State Management |
| 3 | 🔀 Edge Types |
| 4 | 🧠 Memory |
| 5 | 🤖 Node Types |
| 6 | ⚙️ Graph Runtime |
| 7 | 🛠️ Tools Integration |
| 8 | 🚀 Full End-to-End Example |

---
## Section 1 — 🔧 Setup & Secrets

Install dependencies and load your API key from **Colab Secrets** (`Secrets` tab in the left sidebar → add key `OPENAI_API_KEY`).

In [1]:
# Install required packages
!pip install -q langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 14.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

# Load OpenAI API key from Colab Secrets
# Go to: left sidebar → 🔑 Secrets → + Add new secret
# Name: OPENAI_API_KEY   Value: sk-...
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("✅ API key loaded successfully!")

✅ API key loaded successfully!


In [3]:
# Core imports used throughout the notebook
from typing import TypedDict, Annotated, Sequence, Literal
import operator

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Initialize the LLM — GPT-4o Mini
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Imports complete. Using model:", llm.model_name)

✅ Imports complete. Using model: gpt-4o-mini


---
## Section 2 — 🗃️ State Management

LangGraph graphs carry a **State** object between nodes. The three standard fields are:
- `messages` — conversation history (a `List` that auto-appends)
- `memory` — a `Dict` for arbitrary key-value data that persists across turns
- `context` — a `Dict` for run-scoped metadata (user profile, session info, etc.)

In [4]:
# ── State Management ─────────────────────────────────────────────────────────

class AgentState(TypedDict):
    """The central state object shared by every node in the graph."""
    # add_messages is a reducer: new messages are APPENDED, not overwritten
    messages: Annotated[list[BaseMessage], add_messages]
    # memory persists facts the agent has learned
    memory: dict
    # context carries per-run metadata (user id, preferences, etc.)
    context: dict


# ── Demo: create an initial state and inspect it ──────────────────────────────
initial_state: AgentState = {
    "messages": [HumanMessage(content="Hello! What can you do?")],
    "memory": {"user_name": "Alice", "session_count": 1},
    "context": {"user_id": "u_001", "language": "en"},
}

print("=== Initial State ===")
print(f"messages : {[m.content for m in initial_state['messages']]}")
print(f"memory   : {initial_state['memory']}")
print(f"context  : {initial_state['context']}")

=== Initial State ===
messages : ['Hello! What can you do?']
memory   : {'user_name': 'Alice', 'session_count': 1}
context  : {'user_id': 'u_001', 'language': 'en'}


In [5]:
# Demonstrate how add_messages reducer works ──────────────────────────────────
from langgraph.graph.message import add_messages

existing = [HumanMessage(content="First message")]
new_msgs  = [AIMessage(content="First reply"), HumanMessage(content="Follow-up")]

merged = add_messages(existing, new_msgs)   # reducer merges, never replaces
print("Merged message list:")
for m in merged:
    print(f"  [{m.__class__.__name__}] {m.content}")

Merged message list:
  [HumanMessage] First message
  [AIMessage] First reply
  [HumanMessage] Follow-up


---
## Section 3 — 🔀 Edge Types

Edges connect nodes and determine execution flow. LangGraph supports four kinds:

| Edge Type | Description |
|---|---|
| **Direct** | Always goes from Node A → Node B |
| **Conditional** | A function inspects state and picks the next node |
| **Cycle** | Node can route back to an earlier node (loops) |
| **End** | Sends execution to the terminal `END` node |

In [6]:
# ── Edge Types Demo ───────────────────────────────────────────────────────────

class EdgeDemoState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    needs_tool: bool          # flag used by conditional edge
    iteration:  int           # counter used by cycle edge


# Node definitions ─────────────────────────────────────────────────────────────
def start_node(state: EdgeDemoState) -> EdgeDemoState:
    """Entry point — calls GPT-4o Mini."""
    response = llm.invoke(state["messages"])
    needs = "tool" in response.content.lower() or "calculate" in response.content.lower()
    return {"messages": [response], "needs_tool": needs, "iteration": state.get("iteration", 0)}

def tool_node(state: EdgeDemoState) -> EdgeDemoState:
    """Simulates tool execution then loops back if needed."""
    tool_result = AIMessage(content="[Tool result: 42]")
    return {"messages": [tool_result], "needs_tool": False, "iteration": state["iteration"] + 1}

def final_node(state: EdgeDemoState) -> EdgeDemoState:
    """Formats the final answer."""
    last = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"Final answer: {last}")]}


# ── Conditional edge function ──────────────────────────────────────────────────
def route_after_start(state: EdgeDemoState) -> Literal["tool_node", "final_node"]:
    """Conditional edge: if a tool is needed, go to tool_node; otherwise finish."""
    if state["needs_tool"] and state["iteration"] < 2:   # cycle guard
        return "tool_node"
    return "final_node"


# ── Build the graph with all edge types ───────────────────────────────────────
edge_graph = StateGraph(EdgeDemoState)

edge_graph.add_node("start_node", start_node)
edge_graph.add_node("tool_node",  tool_node)
edge_graph.add_node("final_node", final_node)

# DIRECT edge: START → start_node
edge_graph.add_edge(START, "start_node")

# CONDITIONAL edge: start_node → tool_node OR final_node
edge_graph.add_conditional_edges("start_node", route_after_start)

# CYCLE edge: tool_node loops back to start_node
edge_graph.add_edge("tool_node", "start_node")

# END edge: final_node → END
edge_graph.add_edge("final_node", END)

compiled_edge_graph = edge_graph.compile()

# Run it
result = compiled_edge_graph.invoke({
    "messages": [HumanMessage(content="Just say hello, no tools needed.")],
    "needs_tool": False,
    "iteration": 0,
})

print("=== Edge Demo Result ===")
for m in result["messages"]:
    print(f"  [{m.__class__.__name__}] {m.content[:80]}")

=== Edge Demo Result ===
  [HumanMessage] Just say hello, no tools needed.
  [AIMessage] Hello!
  [AIMessage] Final answer: Hello!


---
## Section 4 — 🧠 Memory

LangGraph's memory system provides:
- **Checkpointing** — snapshot state at every step
- **State Persistence** — resume exactly where a run left off
- **History Tracking** — replay the full state timeline
- **Time Travel** — branch from any past checkpoint

In [7]:
# ── Memory: Checkpointing & State Persistence ─────────────────────────────────

class MemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    memory:   dict
    context:  dict

def chat_node(state: MemoryState) -> MemoryState:
    """Simple chat node that also stores the turn count in memory."""
    response = llm.invoke(state["messages"])
    updated_memory = {**state.get("memory", {}),
                      "turn_count": state.get("memory", {}).get("turn_count", 0) + 1}
    return {"messages": [response], "memory": updated_memory}

# MemorySaver stores checkpoints in RAM (swap for SqliteSaver for disk persistence)
checkpointer = MemorySaver()

mem_graph = StateGraph(MemoryState)
mem_graph.add_node("chat", chat_node)
mem_graph.add_edge(START, "chat")
mem_graph.add_edge("chat", END)

mem_app = mem_graph.compile(checkpointer=checkpointer)

# Each run with the SAME thread_id continues the same conversation
config = {"configurable": {"thread_id": "demo-thread-1"}}

# Turn 1
r1 = mem_app.invoke(
    {"messages": [HumanMessage(content="My name is Bob.")], "memory": {}, "context": {}},
    config=config
)
print("Turn 1 reply:", r1["messages"][-1].content[:100])
print("Memory after turn 1:", r1["memory"])

# Turn 2 — memory is automatically restored from checkpoint
r2 = mem_app.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "memory": {}, "context": {}},
    config=config
)
print("\nTurn 2 reply:", r2["messages"][-1].content[:100])
print("Memory after turn 2:", r2["memory"])

Turn 1 reply: Nice to meet you, Bob! How can I assist you today?
Memory after turn 1: {'turn_count': 1}

Turn 2 reply: Your name is Bob. How can I help you today, Bob?
Memory after turn 2: {'turn_count': 1}


In [8]:
# ── History Tracking & Time Travel ────────────────────────────────────────────

# Get the full state history for our thread
history = list(mem_app.get_state_history(config))
print(f"Total checkpoints stored: {len(history)}")
for i, snapshot in enumerate(history):
    turn = snapshot.values.get("memory", {}).get("turn_count", 0)
    msgs = len(snapshot.values.get("messages", []))
    print(f"  Checkpoint {i}: turn_count={turn}, messages={msgs}, id={str(snapshot.config)[:60]}")

# Time Travel — replay from checkpoint 0 (the very first snapshot)
if len(history) >= 2:
    past_config = history[-1].config   # oldest snapshot
    past_state  = mem_app.get_state(past_config)
    print("\n=== Time Travel: state at checkpoint 0 ===")
    print("  turn_count:", past_state.values.get("memory", {}).get("turn_count"))
    print("  messages  :", [m.content[:40] for m in past_state.values.get("messages", [])])

Total checkpoints stored: 6
  Checkpoint 0: turn_count=1, messages=4, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_
  Checkpoint 1: turn_count=0, messages=3, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_
  Checkpoint 2: turn_count=1, messages=2, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_
  Checkpoint 3: turn_count=1, messages=2, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_
  Checkpoint 4: turn_count=0, messages=1, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_
  Checkpoint 5: turn_count=0, messages=0, id={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_

=== Time Travel: state at checkpoint 0 ===
  turn_count: None
  messages  : []


---
## Section 5 — 🤖 Node Types

Every processing step in LangGraph is a **node**. Four common patterns:

| Node Type | Role |
|---|---|
| **LLM Node** | Calls a language model to generate text |
| **Tool Node** | Executes a deterministic function / API call |
| **Router Node** | Inspects state and decides the next route |
| **Action Node** | Runs side-effects (write DB, send email, etc.) |

In [9]:
# ── Node Types Demo ───────────────────────────────────────────────────────────

class NodeDemoState(TypedDict):
    messages:   Annotated[list[BaseMessage], add_messages]
    route:      str    # set by the router node
    action_log: list   # populated by the action node


# 1. LLM Node ──────────────────────────────────────────────────────────────────
def llm_node(state: NodeDemoState) -> NodeDemoState:
    """Calls GPT-4o Mini and appends its response."""
    system = SystemMessage(content="You are a helpful assistant. Be concise.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


# 2. Tool Node ─────────────────────────────────────────────────────────────────
def calculator_tool_node(state: NodeDemoState) -> NodeDemoState:
    """Extracts a math expression from the last message and evaluates it."""
    last = state["messages"][-1].content
    # simple eval guard – only digits and basic operators
    import re
    expr = re.search(r"[\d\+\-\*/\.\(\) ]+", last)
    try:
        answer = eval(expr.group()) if expr else "N/A"  # noqa: S307
    except Exception:
        answer = "Could not evaluate"
    result_msg = AIMessage(content=f"[Calculator] {expr.group() if expr else '?'} = {answer}")
    return {"messages": [result_msg]}


# 3. Router Node ───────────────────────────────────────────────────────────────
def router_node(state: NodeDemoState) -> NodeDemoState:
    """Decides whether the user wants a calculation or a general answer."""
    user_text = state["messages"][0].content.lower()
    keywords  = ["calculate", "compute", "math", "+", "-", "*", "/", "sum"]
    route     = "calculator" if any(k in user_text for k in keywords) else "llm"
    return {"route": route}


# 4. Action Node ───────────────────────────────────────────────────────────────
def action_node(state: NodeDemoState) -> NodeDemoState:
    """Logs the completed interaction (side-effect simulation)."""
    last = state["messages"][-1].content
    log  = state.get("action_log", [])
    log.append({"event": "response_logged", "preview": last[:60]})
    print(f"[Action Node] Logged interaction. Total logs: {len(log)}")
    return {"action_log": log}


# Conditional edge based on router_node output
def dispatch(state: NodeDemoState) -> Literal["calculator_tool_node", "llm_node"]:
    return "calculator_tool_node" if state["route"] == "calculator" else "llm_node"


# Build graph
node_graph = StateGraph(NodeDemoState)
node_graph.add_node("router",     router_node)
node_graph.add_node("llm_node",   llm_node)
node_graph.add_node("calculator_tool_node", calculator_tool_node)
node_graph.add_node("action",     action_node)

node_graph.add_edge(START, "router")
node_graph.add_conditional_edges("router", dispatch)
node_graph.add_edge("llm_node",   "action")
node_graph.add_edge("calculator_tool_node", "action")
node_graph.add_edge("action",     END)

node_app = node_graph.compile()

# Test 1: general question → LLM node
r_llm = node_app.invoke({"messages": [HumanMessage(content="What is the capital of France?")],
                          "route": "", "action_log": []})
print("General question →", r_llm["messages"][-1].content[:80])
print()

# Test 2: math question → Tool node
r_calc = node_app.invoke({"messages": [HumanMessage(content="Calculate 123 * 456")],
                           "route": "", "action_log": []})
print("Math question   →", r_calc["messages"][-1].content)

[Action Node] Logged interaction. Total logs: 1
General question → The capital of France is Paris.

[Action Node] Logged interaction. Total logs: 1
Math question   → [Calculator]  123 * 456 = 56088


---
## Section 6 — ⚙️ Graph Runtime

The runtime pipeline is:
1. **Compilation** — validate graph, lock in node/edge schema
2. **Execution** — run nodes in topological order
3. **State Updates** — each node returns a partial state; reducers merge it
4. **Event Streaming** — stream individual steps as they complete

In [10]:
# ── Graph Runtime: Compilation & Execution ────────────────────────────────────

class RuntimeState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    step_log: list

def step_a(state: RuntimeState) -> RuntimeState:
    return {"messages": [AIMessage(content="Step A: context gathered")],
            "step_log": state.get("step_log", []) + ["step_a"]}

def step_b(state: RuntimeState) -> RuntimeState:
    """Calls GPT-4o Mini with the full message history."""
    response = llm.invoke(state["messages"])
    return {"messages": [response],
            "step_log": state["step_log"] + ["step_b"]}

def step_c(state: RuntimeState) -> RuntimeState:
    return {"messages": [AIMessage(content="Step C: post-processing complete")],
            "step_log": state["step_log"] + ["step_c"]}

# ── Compilation ────────────────────────────────────────────────────────────────
runtime_graph = StateGraph(RuntimeState)
runtime_graph.add_node("step_a", step_a)
runtime_graph.add_node("step_b", step_b)
runtime_graph.add_node("step_c", step_c)
runtime_graph.add_edge(START, "step_a")
runtime_graph.add_edge("step_a", "step_b")
runtime_graph.add_edge("step_b", "step_c")
runtime_graph.add_edge("step_c", END)

runtime_app = runtime_graph.compile()   # ← Compilation step
print("✅ Graph compiled successfully")

# ── Execution ──────────────────────────────────────────────────────────────────
final = runtime_app.invoke({
    "messages": [HumanMessage(content="Tell me one fun fact about space.")],
    "step_log": []
})
print("\nExecution order:", final["step_log"])
print("LLM response   :", final["messages"][-2].content[:120])  # step_b output

✅ Graph compiled successfully

Execution order: ['step_a', 'step_b', 'step_c']
LLM response   : One fun fact about space is that it is completely silent! Unlike on Earth, where sound travels through air, space is a v


In [11]:
# ── Event Streaming ────────────────────────────────────────────────────────────
print("=== Streaming events step-by-step ===")

for event in runtime_app.stream({
    "messages": [HumanMessage(content="Name one planet.")],
    "step_log": []
}):
    for node_name, node_output in event.items():
        msgs = node_output.get("messages", [])
        latest = msgs[-1].content[:60] if msgs else "(no message)"
        print(f"  📦 [{node_name}] → {latest}")

=== Streaming events step-by-step ===
  📦 [step_a] → Step A: context gathered
  📦 [step_b] → Mars.
  📦 [step_c] → Step C: post-processing complete


---
## Section 7 — 🛠️ Tools Integration

LangGraph integrates tools in four phases:
- **Tool Binding** — attach tools to the LLM so it knows what's available
- **Tool Execution** — run the chosen tool when the LLM requests it
- **Error Handling** — catch tool failures gracefully and recover
- **Result Processing** — inject tool output back into state as a message

In [12]:
from langchain_core.messages import ToolMessage
import json

# ── Define Tools ───────────────────────────────────────────────────────────────
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city (mock data)."""
    mock_db = {
        "london": "12°C, overcast",
        "new york": "22°C, sunny",
        "tokyo": "18°C, partly cloudy",
    }
    return mock_db.get(city.lower(), f"No data for {city}")

@tool
def calculate(expression: str) -> str:
    """Evaluates a basic arithmetic expression and returns the result."""
    try:
        # safe eval: only allow numbers and operators
        import re
        if not re.match(r'^[\d\s\+\-\*/\.\(\)]+$', expression):
            return "Error: unsafe expression"
        return str(eval(expression))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"

tools = [get_weather, calculate]

# ── Tool Binding ───────────────────────────────────────────────────────────────
llm_with_tools = llm.bind_tools(tools)
print("✅ Tools bound:", [t.name for t in tools])


# ── Tool Execution & Result Processing ────────────────────────────────────────
class ToolState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

TOOL_MAP = {t.name: t for t in tools}

def llm_with_tools_node(state: ToolState) -> ToolState:
    """Calls GPT-4o Mini with tool bindings."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_executor_node(state: ToolState) -> ToolState:
    """Executes all tool calls requested by the LLM and injects results."""
    last_msg  = state["messages"][-1]
    results   = []

    for tool_call in last_msg.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id   = tool_call["id"]

        # ── Error Handling ────────────────────────────────────────────────────
        try:
            if tool_name not in TOOL_MAP:
                raise ValueError(f"Unknown tool: {tool_name}")
            output = TOOL_MAP[tool_name].invoke(tool_args)
            print(f"  🔧 Executed '{tool_name}' → {output}")
        except Exception as e:
            output = f"[Error] {e}"
            print(f"  ❌ Tool error: {e}")

        # ── Result Processing ──────────────────────────────────────────────────
        results.append(ToolMessage(content=str(output), tool_call_id=tool_id))

    return {"messages": results}

def should_use_tools(state: ToolState) -> Literal["tool_executor_node", "__end__"]:
    last = state["messages"][-1]
    return "tool_executor_node" if getattr(last, "tool_calls", None) else "__end__"


# Build tool graph
tool_graph = StateGraph(ToolState)
tool_graph.add_node("llm",          llm_with_tools_node)
tool_graph.add_node("tool_executor_node", tool_executor_node)
tool_graph.add_edge(START, "llm")
tool_graph.add_conditional_edges("llm", should_use_tools)
tool_graph.add_edge("tool_executor_node", "llm")   # loop back after tool execution

tool_app = tool_graph.compile()

# Run examples
print("\n=== Tool Integration Examples ===")

r = tool_app.invoke({"messages": [HumanMessage(content="What's the weather in Tokyo and what is 15 * 8?")]})
print("Final reply:", r["messages"][-1].content[:200])

✅ Tools bound: ['get_weather', 'calculate']

=== Tool Integration Examples ===
  🔧 Executed 'get_weather' → 18°C, partly cloudy
  🔧 Executed 'calculate' → 120
Final reply: The weather in Tokyo is 18°C and partly cloudy. Additionally, the result of 15 * 8 is 120.


---
## Section 8 — 🚀 Full End-to-End Example

Putting it all together: a **persistent conversational agent** that uses:
- `AgentState` with messages + memory + context
- All four node types (LLM, Tool, Router, Action)
- Conditional + cycle edges
- MemorySaver checkpointing
- Tool binding + execution + error handling
- Event streaming

In [15]:
from typing import TypedDict, Annotated, Literal

from langchain_core.messages import (
    BaseMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# -------------------------------------------------------------------
# STATE
# -------------------------------------------------------------------

class FullState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    memory: dict
    context: dict
    action_log: list


# -------------------------------------------------------------------
# TOOLS
# -------------------------------------------------------------------

@tool
def remember_fact(key: str, value: str) -> str:
    """Store a key-value fact for later retrieval."""
    return f"Remembered: {key} = {value}"


@tool
def get_weather(city: str) -> str:
    """Return mock weather data."""
    weather = {
        "paris": "15°C, rainy",
        "berlin": "10°C, cloudy",
        "sydney": "28°C, sunny",
    }
    return weather.get(city.lower(), f"No weather found for {city}")


TOOLS = [remember_fact, get_weather]
TOOL_MAP = {tool.name: tool for tool in TOOLS}

# Assume llm is already created, for example:
#
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")
#
llm_with_tools = llm.bind_tools(TOOLS)


# -------------------------------------------------------------------
# LLM NODE
# -------------------------------------------------------------------

def llm_node(state: FullState):

    system_prompt = SystemMessage(
        content=(
            "You are a helpful assistant.\n"
            "If the user tells you important personal information "
            "(name, preferences, etc.), call remember_fact.\n"
            "If the user asks about weather, call get_weather."
        )
    )

    response = llm_with_tools.invoke(
        [system_prompt] + state["messages"]
    )

    return {
        "messages": [response]
    }


# -------------------------------------------------------------------
# TOOL NODE
# -------------------------------------------------------------------

def tool_node(state: FullState):

    last_message = state["messages"][-1]

    memory = dict(state.get("memory", {}))
    tool_messages = []

    for tool_call in last_message.tool_calls:

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        try:
            result = TOOL_MAP[tool_name].invoke(tool_args)

            # Persist facts into graph state
            if tool_name == "remember_fact":
                memory[tool_args["key"]] = tool_args["value"]

        except Exception as e:
            result = f"Tool Error: {e}"

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
            )
        )

    return {
        "messages": tool_messages,
        "memory": memory,
    }


# -------------------------------------------------------------------
# ACTION NODE
# -------------------------------------------------------------------

def action_node(state: FullState):

    logs = list(state.get("action_log", []))

    logs.append(
        {
            "turn": len(logs) + 1,
            "memory": dict(state.get("memory", {})),
            "last_message": state["messages"][-1].content,
        }
    )

    return {
        "action_log": logs
    }


# -------------------------------------------------------------------
# ROUTER
# -------------------------------------------------------------------

def route_after_llm(
    state: FullState,
) -> Literal["tools", "action"]:

    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return "action"


# -------------------------------------------------------------------
# BUILD GRAPH
# -------------------------------------------------------------------

graph_builder = StateGraph(FullState)

graph_builder.add_node("llm", llm_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("action", action_node)

graph_builder.add_edge(START, "llm")

graph_builder.add_conditional_edges(
    "llm",
    route_after_llm,
)

# After tools run, go back to the LLM
graph_builder.add_edge("tools", "llm")

# Finish after action node
graph_builder.add_edge("action", END)


# -------------------------------------------------------------------
# COMPILE
# -------------------------------------------------------------------

checkpointer = MemorySaver()

app = graph_builder.compile(
    checkpointer=checkpointer
)

print("✅ Graph compiled successfully!")

✅ Graph compiled successfully!


In [19]:
from langchain_core.messages import HumanMessage, AIMessage

# ------------------------------------------------------------------
# Initial State (created only once)
# ------------------------------------------------------------------

state: FullState = {
    "messages": [],
    "memory": {},
    "context": {
        "user": "demo"
    },
    "action_log": [],
}

# Optional: thread ID for checkpointing
agent_config = {
    "configurable": {
        "thread_id": "full-agent-demo"
    }
}


# ------------------------------------------------------------------
# Chat Function
# ------------------------------------------------------------------

def chat(user_input: str):
    global state

    # Add the user's message to the conversation
    state["messages"].append(
        HumanMessage(content=user_input)
    )

    # Invoke the graph using the CURRENT state
    state = app.invoke(
        state,
        config=agent_config,
    )

    # Find the most recent AI response
    last_ai = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage):
            last_ai = msg
            break

    print("=" * 60)
    print(f"🧑 You: {user_input}")
    print()

    if last_ai:
        print(f"🤖 AI: {last_ai.content}")

    print()
    print("📝 Memory:")
    print(state["memory"])

    print()
    print("📦 Context:")
    print(state["context"])

    print()
    print(f"💬 Total Messages: {len(state['messages'])}")

    return state


# ------------------------------------------------------------------
# Multi-turn Conversation
# ------------------------------------------------------------------

chat("Please remember that my favourite city is Paris.")

chat("What's the weather like in my favourite city?")

chat("Summarise what you know about me so far.")

🧑 You: Please remember that my favourite city is Paris.

🤖 AI: I've noted that your favorite city is Paris!

📝 Memory:
{'favourite_city': 'Paris'}

📦 Context:
{'user': 'demo'}

💬 Total Messages: 4
🧑 You: What's the weather like in my favourite city?

🤖 AI: The weather in Paris is currently 15°C and rainy.

📝 Memory:
{'favourite_city': 'Paris'}

📦 Context:
{'user': 'demo'}

💬 Total Messages: 8
🧑 You: Summarise what you know about me so far.

🤖 AI: So far, I know that your favorite city is Paris.

📝 Memory:
{'favourite_city': 'Paris'}

📦 Context:
{'user': 'demo'}

💬 Total Messages: 10


{'messages': [HumanMessage(content='Please remember that my favourite city is Paris.', additional_kwargs={}, response_metadata={}, id='8535f447-3f1e-482a-ba4d-fcd882433ec3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 114, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1aad5391f', 'id': 'chatcmpl-DtG2jmGy9DRzudQyH7VVCjUtXiX9A', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eeb2d-6428-7b02-961a-114b9b50536a-0', tool_calls=[{'name': 'remember_fact', 'args': {'key': 'favourite_city', 'value': 'Paris'}, 'id': 'call_w31fHYVE1Mz7Q0boNnybVcmX', 'type': 'tool_call'}], invalid_tool_call

In [22]:
# ── Stream the final turn ──────────────────────────────────────────────────────
print("=== Streaming events for final turn ===")
for event in app.stream(
    {**base_state, "messages": [HumanMessage(content="What tools do you have?")]},
    config={"configurable": {"thread_id": "full-agent-stream"}}
):
    for node_name, output in event.items():
        msgs = output.get("messages", [])
        preview = msgs[-1].content[:70] if msgs else "-"
        print(f"  [{node_name}] → {preview}")

=== Streaming events for final turn ===
  [llm] → I have access to two main tools:

1. **remember_fact**: This tool allo
  [action] → -


---
## 🎉 Summary

| Concept | What we built |
|---|---|
| **State Management** | `AgentState` with `messages`, `memory`, `context` using reducers |
| **Edge Types** | Direct, Conditional, Cycle, and End edges in a live graph |
| **Memory** | `MemorySaver` checkpointing + state history + time travel |
| **Node Types** | LLM, Tool, Router, and Action nodes each with a distinct role |
| **Graph Runtime** | Compilation, execution, state updates, event streaming |
| **Tools Integration** | Tool binding, execution loop, error handling, result processing |

All of this runs on **GPT-4o Mini** with the API key loaded from Colab Secrets. 🚀